# Choosing feeds

A place is often served by several feeds, and many of them carry the same
buses and trains. This page shows how to list a place's feeds, compare them
side by side, and let `transitio` recommend which ones to fetch, for a place or
for an area of your own.

## A place's feeds

`feeds()` lists the feeds the index records for a place. Let's list
Augsburg's local and regional feeds:

In [1]:
import transitio

augsburg = transitio.place("Augsburg")
for feed in augsburg.feeds(tiers=["local", "regional"]):
    print(feed.feed_id, sorted(feed.tiers), feed.relevance_category, feed.service_start, feed.service_end)

f-mdb-3215 ['local', 'regional'] primary 2026-10-17 2026-12-12
f-germany~urban~transport ['local', 'regional'] primary 2026-10-31 2026-11-30
f-germany~regional~rail ['local', 'regional'] primary 2026-10-31 2026-11-30
f-u281z9-mvv ['local', 'regional'] primary 2026-11-01 2027-01-31
f-mdb-1139 ['local', 'regional'] primary 2018-12-09 2019-12-14
f-mdb-857 ['local'] primary 2024-12-15 2025-12-31
f-nvbw~ding ['regional'] secondary 2026-07-12 2026-12-12
f-nvbw~oam ['regional'] secondary 2026-07-12 2026-12-12
f-mdb-2393 ['regional'] secondary 2026-08-19 2026-12-12
f-nvbw~bodo ['regional'] secondary 2026-07-12 2026-12-12
f-mdb-2138 ['regional'] secondary 2023-12-10 2024-12-14


Each feed also carries its relevance (`relevance_category` and `relevance`),
whether it has route shapes (`has_shapes`), its `license` and its
GTFS-realtime companions (`realtime`), the live-data feeds that go with a
static timetable.

Without arguments, `feeds()` lists every feed serving the place, of every
tier and relevance category, cross-border ones included.
`categories="default"` gives the view for the kind of place:

- a city or a metro: its primary and secondary feeds, that is, its local and
  regional service;
- a region: its secondary and tertiary feeds;
- a country: its tertiary feeds.

In that view, a region or country of at most 1,000 km², such as Monaco or
San Juan, counts as town-sized and keeps its primary, secondary and tertiary
feeds. `international=False` leaves out the cross-border feeds.

You can also filter the list: `exclude` drops the named tiers, and
`requires=["shapes.txt"]` keeps only the feeds that carry those files.
`feed.realtime` lists the GTFS-realtime companions tied to a static feed,
and `Index.realtime_unlinked()` lists the companions the index could not tie
to one.

When `fetch(place=...)` without `tiers` finds no feed at the tiers
`recommend()` considers for the kind of place (see below), although feeds at
other tiers serve it, `result.view_note` names those feeds and the `tiers`
that would fetch them, and a warning repeats the note.

To compare a place's feeds side by side, take them as a table. Tallinn is
served by several feeds:

In [2]:
tallinn = transitio.place("Tallinn")
table = tallinn.feeds(categories=None).to_dataframe()
table[["feed_id", "name", "covers", "repeats", "stop_count", "reason"]]

,feed_id,name,covers,repeats,stop_count,reason
0,f-ud-tallinn,TLT,0.905817,f-mdb-3235 100 %,18052,primary (local tier): 45 % of the departures s...
1,f-mdb-3235,Ühistranspordiregistri avaandmed,0.999775,"f-ud-tallinn 90 %, f-ud-harjumaa 8 %, f-ud-kau...",18288,"primary (international, local, national, regio..."
2,f-ud-harjumaa,Harjumaa,0.084983,"f-mdb-3235 100 %, f-ud-kaugliinid 2 %",18280,primary (local tier): 4 % of the departures su...
3,f-ud-elron,ELRON,0.006519,f-mdb-3235 100 %,18054,"primary (international, local, national, regio..."
4,f-ud-raplamaa,Raplamaa,0.000369,f-mdb-3235 100 %,18285,primary (local tier): 0.018 % of the departure...
5,f-ud-laanemaa,Läänemaa,0.000050,f-mdb-3235 100 %,18288,primary (local tier): 0.0026 % of the departur...
6,f-ud-kaugliinid,Siseriiklikud kaugbussiliinid,0.001910,"f-mdb-3235 100 %, f-ud-harjumaa 6 %",18071,"primary (local, national tiers): 0.11 % of the..."
7,f-ud-laanevirumaa,Lääne-Virumaa,0.000089,f-mdb-3235 100 %,18288,tertiary (national tier): 0.0041 % of the depa...
8,f-ud-parnumaa,Pärnumaa,0.000098,f-mdb-3235 100 %,18071,tertiary (national tier): 0.0054 % of the depa...
9,f-mdb-tdg-11681,Réseau européen FlixBus et FlixTrain,0.000170,f-u-flixbus 100 %,2077,"tertiary (international, national tiers): 0.00..."


The table has one row per feed, with each feed's tiers, relevance, service,
modes, timetable window, access and stop count, among other columns. These
columns help most when you compare feeds:

- `share_of_place` is the feed's share of the departures summed over the
  place's feeds, or its share of the stops where the index measured stops.
- `covers` is the share of the place's departures the feed runs, each
  departure counted once.
- `repeats` names the other feeds that also run the feed's departures.
- `stale_when_indexed` gives the day the feed's timetable ended, when that was
  more than 30 days before the index crawled the feed.
- `contained_in` names the larger feeds whose stops and routes contain this
  feed's.
- `reason` sums up the row in words.

`covers` and `repeats` need an index that records which feeds run the same
lines; without that evidence, both are empty.

## Which feeds to use

Several feeds often run the same service: a city's own feed, a national
aggregate and a regional one may each carry nearly all of a city's
departures. `recommend()` says which feeds to take for a study day and why it
leaves out the others. Give it a day the place's feeds run, here Wednesday
11 November 2026:

In [3]:
rec = tallinn.recommend("2026-11-11")
print(rec)

Tallinn (city), 2026-11-11: take 5 feeds, covering about 100 % of the departures the index records there, each counted once
  considers local and regional feeds only, and those of unknown tier; tiers=None adds national and international feeds
  + f-ud-tallinn (TLT): covers 90 % of the place's departures (90 % of bus; 92 % of rail, subway and tram)
  + f-ud-elron (ELRON): adds rail, subway and tram the feeds above lack: 7 % of the place's rail, subway and tram
  + f-ud-kaugliinid (Siseriiklikud kaugbussiliinid): adds bus the feeds above lack: 0.021 % of the place's bus
  + f-ud-harjumaa (Harjumaa): adds bus the feeds above lack: 9 % of the place's bus
  + f-mdb-3235 (Ühistranspordiregistri avaandmed): adds bus the feeds above lack: 9 % of the place's bus
  - f-ud-raplamaa, f-ud-laanemaa: repeats f-mdb-3235 (100 % of its departures)
  - f-mdb-tdg-11681, f-u-flixbus, f-germany~urban~transport: not considered: serves the place at the national and international tiers
  - f-ud-parnumaa, f-ud

As we can see, `recommend()` takes five feeds for Tallinn: the city's own
feed, the rail feed, the long-distance and Harju County bus feeds and, for
the departures they lack, the national feed. It leaves out the Rapla and
Lääne County feeds, whose departures the national feed also runs, and the
feeds that serve Tallinn only at the national and international tiers. The candidates are
the feeds serving the place at the tiers of its kind, whatever their
category, cross-border ones included: local and regional feeds, and those
of unknown tier, at a city or a metro; every tier at a region or a country.
`tiers=` names other tiers, and `tiers=None` takes every tier.
`recommend()` then chooses in two steps:

1. **It leaves out the feeds that cannot serve the study day.** These are
   the feeds whose timetable as indexed does not run on the study day,
   those that need a paid account, and those the index measured no service
   for. Without a study day no feed is left out for its dates, since older
   data serves research as well; the feeds whose timetable runs today are
   then taken first. Feeds the index marks as private or campus services or
   unofficial compilations are listed as not considered.
2. **It takes the candidates the most local first:** the one covering the
   smallest area, then the open one, the newer timetable and the fewer
   stops. Counting each of the place's departures once, it takes a feed when
   it adds a departure the feeds taken before it do not run, so an
   operator's own feed comes before an aggregate repeating it, and the
   aggregate is taken for what it adds. `target=` stops the selection once
   the taken feeds cover that share of the departures and 80 % of each
   mode's, with rail, subway and tram counting as one mode, and
   `max_feeds=` once that many are taken.

Step 2 needs an index that records which feeds run the same lines. Without
that evidence, `recommend()` takes every candidate. An index merged from
several builds compares feeds only within the group measured together;
the candidates measured apart from that group are taken without
comparison, and `fetch` later leaves out the trips they repeat.

The recommendation's `feed_ids` lists the taken feeds, and `to_dataframe()`
gives every feed with the reason it was taken or left out and its timetable
window as indexed:

In [4]:
print(rec.feed_ids)
rec.to_dataframe()

['f-ud-tallinn', 'f-ud-elron', 'f-ud-kaugliinid', 'f-ud-harjumaa', 'f-mdb-3235']


,feed_id,name,taken,adds,reason,service_start,service_end
0,f-ud-tallinn,TLT,True,0.907757,covers 90 % of the place's departures (90 % of...,2026-09-12,2026-12-31
1,f-ud-elron,ELRON,True,0.006501,"adds rail, subway and tram the feeds above lac...",2026-08-31,2026-12-31
2,f-ud-kaugliinid,Siseriiklikud kaugbussiliinid,True,0.000188,adds bus the feeds above lack: 0.021 % of the ...,2025-09-01,2031-11-14
3,f-ud-harjumaa,Harjumaa,True,0.084976,adds bus the feeds above lack: 9 % of the plac...,2026-08-01,2026-12-31
4,f-mdb-3235,Ühistranspordiregistri avaandmed,True,0.092243,adds bus the feeds above lack: 9 % of the plac...,2025-01-01,2029-12-31
5,f-ud-raplamaa,Raplamaa,False,0.000000,repeats f-mdb-3235 (100 % of its departures),2026-10-26,2026-12-23
6,f-ud-laanemaa,Läänemaa,False,0.000000,repeats f-mdb-3235 (100 % of its departures),2026-11-02,2026-12-23
7,f-mdb-tdg-11681,Réseau européen FlixBus et FlixTrain,False,NaN,not considered: serves the place at the nation...,2026-11-04,2027-05-03
8,f-u-flixbus,"Greyhound, Flixbus",False,NaN,not considered: serves the place at the nation...,2026-11-01,2027-04-30
9,f-ud-parnumaa,Pärnumaa,False,NaN,not considered: serves the place at the nation...,2025-10-01,2027-09-26


To fetch only the feeds it takes, pass the recommendation to `fetch`,
which takes the place and the day from it:

```python
result = transitio.fetch(feeds=rec)
```

[Fetching data](fetching_data.ipynb) shows this call and its options.

## Feeds for an area

`transitio.index.area` finds the index places that cover an area of your
own, such as a study region drawn as a polygon. Let's combine Munich and the
district around it into one study region:

In [5]:
import shapely

munich = transitio.place("Munich", kind="city")
district = transitio.place("Munich District")
region = shapely.union_all([munich.geometry, district.geometry])
area = transitio.index.area(region)
area.coverage          # the share of the area's land the parts cover

1.0

`area.parts` lists the places the area is made of:

In [6]:
for part in area.parts:
    print(part.place, part.whole, round(part.inside, 2), round(part.holds, 2))

Place(tp_37, city, 'Munich') True 1.0 0.32
Place(tp_928, region, 'Munich District') True 1.0 0.06
Place(tp_20102, city, 'Sauerlach') True 1.0 0.06
Place(tp_14136, city, 'Aying') True 1.0 0.06
Place(tp_14560, city, 'Ismaning') True 1.0 0.04
Place(tp_14138, city, 'Brunnthal') True 1.0 0.04
Place(tp_21554, city, 'Aubing-Lochhausen-Langwied') True 1.0 0.03
Place(tp_19802, city, 'Oberhaching') True 1.0 0.03
Place(tp_14473, city, 'Oberschleißheim') True 1.0 0.03
Place(tp_14388, city, 'Feldmoching-Hasenbergl') True 1.0 0.03
Place(tp_19476, city, 'Straßlach-Dingharting') True 1.0 0.03
Place(tp_15092, city, 'Garching bei München') True 1.0 0.03
Place(tp_19553, city, 'Aschheim') True 1.0 0.03
Place(tp_14915, city, 'Grasbrunn') True 1.0 0.03
Place(tp_14408, city, 'Schwabing-Freimann') True 1.0 0.03
Place(tp_19800, city, 'Höhenkirchen-Siegertsbrunn') True 1.0 0.02
Place(tp_16596, city, 'Bogenhausen') True 1.0 0.02
Place(tp_21799, city, 'Trudering-Riem') True 1.0 0.02
Place(tp_14736, city, 'Taufkir

Notice that Munich and its district are parts of the area, and so are the
district's towns and Munich's boroughs, each a place of its own, while
Bavaria, which contains them all, is not a part. `area` chooses the parts by
these rules:

- Only cities, regions and countries with feeds count; metros do not.
- A city is a part when at least half of it lies inside the area, a
  **whole part**, or when it holds at least 1 % of the area.
- A region or a country is a part only for the area's land that no city part
  and no narrower place with feeds covers, when that land is at least 1 % of
  the area. This is why Bavaria is not a part here, while Munich District
  stands for its land outside the towns.

`area.unused` lists the places with feeds that meet the area but are not
parts, with the reason. In the loop above, `inside` is the share of the
place inside the area, and `holds` is the share of the area inside the
place, or, for a region or a country, the share it stands for. `coverage` divides the
area the parts cover by the area's land, that is, its part inside the
index's countries: sea does not count, and land in a country without feeds
counts as uncovered. `country="DE"` keeps only the places of one country.

`area.feeds()` asks each part for its feeds, as `place.feeds()` does and
with the same arguments, and lists each feed once:

In [7]:
[feed.feed_id for feed in area.feeds()]

['f-mdb-2333',
 'f-mdb-3215',
 'f-u281z9-mvv',
 'f-germany~urban~transport',
 'f-germany~regional~rail',
 'f-mdb-2393',
 'f-oebb~at',
 'f-mdb-1139',
 'f-mdb-2138',
 'f-mdb-779',
 'f-germany~long~distance~rail',
 'f-mdb-tdg-11681',
 'f-u-flixbus',
 'f-nvbw~ding',
 'f-nvbw~bodo',
 'f-srq-showmebus',
 'f-nvbw~oam',
 'f-sta~altoadige',
 'f-sru1-livnobus',
 'f-mdb-2900',
 'f-mdb-3261',
 'f-nvbw~bwsbahnubahn',
 'f-mdb-tdg-52605',
 'f-u0-ouibus',
 'f-nvbw~vvs',
 'f-nvbw~rvf',
 'f-nvbw~vhb',
 'f-nvbw~vgf',
 'f-mdb-1859',
 'f-nvbw~tgo',
 'f-mdb-686',
 'f-u-nmbs~sncb',
 'f-nvbw~hnv',
 'f-nvbw~naldo',
 'f-mdb-771',
 'f-nvbw~move',
 'f-polish~trains',
 'f-sncf~fr',
 'f-mdb-tdg-83634',
 'f-bus~dft~gov~uk',
 'f-u0-switzerland',
 'f-u-nl',
 'f-u-se',
 'f-mdb-2939',
 'f-mdb-1258',
 'f-mdb-tdg-83675']

A box inside a city gives the city and the districts the index holds for it,
each a place of its own; a district's `parent` names the city it lies in.
Let's take the centre of Turku:

In [8]:
centre = transitio.index.area((22.20, 60.42, 22.32, 60.48))
print(len(centre.parts), "parts:", [part.place.name for part in centre.parts[:5]])
print(centre.parts[1].place, centre.parts[1].place.parent)

41 parts: ['Turku', 'Pitkämäki', 'Nummi', 'Satama', 'Vasaramäki']
Place(tp_51867, city, 'Pitkämäki') Place(tp_17305, city, 'Turku')


As we can see, the box over the centre of Turku gives Turku itself and 40 of
its districts.

`area.recommend()` works like `place.recommend()`, summing each feed's
departures over the parts of the area.

`fetch` uses the same parts when you give it an area:

- When the parts cover at least half of the area's land, `fetch` takes their
  feeds, with the same `tiers`, `exclude` and `on_unknown` as for a place,
  and `result.places` lists the parts.
- Otherwise, it searches the Mobility Database catalogue by bounding box and
  warns, for example `"the feed index's places cover 20% of the area; 3 feeds
  from the Mobility Database catalogue by bounding box"`.
- With `index=False`, it searches the catalogue without the index and without
  the warning.

The OpenStreetMap extract covers the area itself.

```python
result = transitio.fetch(region, tiers=["local", "regional"], osm=False)
[place.name for place in result.places]   # the parts listed above
```